#### Read data from data/raw

In [2]:
import pandas as pd

# Load the consolidated Excel file into a DataFrame
data_2017_2020 = pd.read_excel("../data/raw/nhanes_raw_merged_2017_2020.xlsx")
data_2021_2023 = pd.read_excel("../data/raw/nhanes_raw_merged_2021_2023.xlsx")

In [3]:
print(data_2017_2020.columns)

Index(['SEQN', 'DMDEDUC2', 'RIAGENDR', 'RIDAGEYR', 'INDFMPIR', 'DR1TCAFF',
       'DR1TSUGR', 'DR1TFIBE', 'DR1TCARB', 'DR1TKCAL', 'DR1TMFAT', 'DR1TPFAT',
       'DR1TSFAT', 'DR1TSODI', 'DR1TPOTA', 'DR1TVC', 'DR1TATOC', 'DR1TPROT',
       'DR1TVB12', 'DR1TCALC', 'DR1TVB6', 'DR2TCAFF', 'DR2TSUGR', 'DR2TFIBE',
       'DR2TCARB', 'DR2TKCAL', 'DR2TMFAT', 'DR2TPFAT', 'DR2TSFAT', 'DR2TSODI',
       'DR2TPOTA', 'DR2TVC', 'DR2TATOC', 'DR2TPROT', 'DR2TVB12', 'DR2TCALC',
       'DR2TVB6', 'BMXBMI', 'BPQ020', 'BPQ030', 'HIQ011', 'PAD680', 'PAQ670',
       'ALQ121', 'SMQ040', 'SMQ020', 'SLD012', 'SLD013', 'OCD150', 'OCQ180',
       'LBXSAL', 'LBXSCR', 'LBXSGL', 'LBXSAPSI', 'LBXHSCRP', 'LBXLYPCT',
       'LBXMCVSI', 'LBXRDW', 'LBXWBCSI', 'LBDTCSI', 'CICLO'],
      dtype='object')


In [4]:
print(data_2021_2023.columns)

Index(['SEQN', 'DMDEDUC2', 'RIAGENDR', 'RIDAGEYR', 'INDFMPIR', 'DR1TCAFF',
       'DR1TSUGR', 'DR1TFIBE', 'DR1TCARB', 'DR1TKCAL', 'DR1TMFAT', 'DR1TPFAT',
       'DR1TSFAT', 'DR1TSODI', 'DR1TPOTA', 'DR1TVC', 'DR1TATOC', 'DR1TPROT',
       'DR1TVB12', 'DR1TCALC', 'DR1TVB6', 'DR2TCAFF', 'DR2TSUGR', 'DR2TFIBE',
       'DR2TCARB', 'DR2TKCAL', 'DR2TMFAT', 'DR2TPFAT', 'DR2TSFAT', 'DR2TSODI',
       'DR2TPOTA', 'DR2TVC', 'DR2TATOC', 'DR2TPROT', 'DR2TVB12', 'DR2TCALC',
       'DR2TVB6', 'BMXBMI', 'BPQ020', 'BPQ030', 'HIQ011', 'PAD680', 'PAD790Q',
       'PAD790U', 'ALQ121', 'SMQ040', 'SMQ020', 'SLD012', 'SLD013', 'OCD150',
       'OCQ180', 'LBXSAL', 'LBXSCR', 'LBXSGL', 'LBXSAPSI', 'LBXHSCRP',
       'LBXLYPCT', 'LBXMCVSI', 'LBXRDW', 'LBXWBCSI', 'LBDTCSI', 'CICLO'],
      dtype='object')


## Identifying which columns differ

-> all the columns represent the same variables in both datasets (with the same measurement units), except the ones that refer to moderate physical acitivity: the combination of the columns 'PAD790Q' and 'PAD790U' in 2021-2023 represents the same variable as the column 'PAQ670' in 2017-2020, but they are expressed in different units.

In [5]:
# ---------- 2021-2023: PAD790Q + PAD790U -> vezes por ano ----------
df21 = data_2021_2023.copy()

q = df21["PAD790Q"]
codigos21 = [7777, 9999]                      # Refused / Don't know

# Unidade: apanha a letra mesmo que venha como "W" ou "b'W'"; ausente -> <NA>
u = df21["PAD790U"].astype("string").str.upper().str.extract(r"([DWMY])")[0]
fator = u.map({"D": 365, "W": 52, "M": 12, "Y": 1})

convertido = q * fator
# Onde q é um código, mantém o código; nos restantes, usa o valor convertido
df21["PAMODYR"] = convertido.where(~q.isin(codigos21), q)
df21 = df21.drop(columns=["PAD790Q", "PAD790U"])

#----------------------------

df17 = data_2017_2020.copy()

dias = df17["PAQ670"]
codigos17 = [77, 99]                          # Refused / Don't know

convertido = dias * 52
df17["PAMODYR"] = convertido.where(~dias.isin(codigos17), dias)
df17 = df17.drop(columns=["PAQ670"])

### Now we can finally append the cycles

In [6]:
df_all = pd.concat([df17, df21], ignore_index=True, sort=False)

In [7]:
len(df_all)

27493

## First cleaning layer
-> Remove participants who are not adults (age below 18).

In [8]:
df_filt = df_all.copy()  #to keep it safe

df_filt = df_filt[df_filt["RIDAGEYR"] >= 20]
df_filt.shape

(17041, 61)

## Second cleaning layer 
-> Remove participants with missing values in the variables required to compute `PhenoAge`.

In [9]:
pheno_columns = ['LBXSAL', 'LBXSCR', 'LBXSGL','LBXSAPSI', 'LBXHSCRP', 'LBXLYPCT', 'LBXMCVSI', 'LBXRDW', 'LBXWBCSI']

In [10]:
df_filt = df_filt.dropna(subset=pheno_columns)
df_filt.shape

(13251, 61)

### Computing PhenoAge (Levine et al., 2018)
-> After computing phenoage discard every variable in `pheno_columns` array.

In [11]:
import numpy as np

def calcular_phenoage(df):
    xb = (
        -19.9067
        - 0.0336 * (df["LBXSAL"] * 10)   #albumin
        + 0.0095 * (df["LBXSCR"] * 88.4)  #creatinine
        + 0.1953 * (df["LBXSGL"] * 0.0555)  #glucose
        + 0.0954 * np.log(df["LBXHSCRP"] / 10)  #C-Reactive Protein
        - 0.0120 * df["LBXLYPCT"]  #Lymphocyte
        + 0.0268 * df["LBXMCVSI"]  #mean cell volume
        + 0.3306 * df["LBXRDW"]
        + 0.0019 * df["LBXSAPSI"]
        + 0.0554 * df["LBXWBCSI"]
        + 0.0804 * df["RIDAGEYR"]
    )
    # M = 1 - exp(-1.51714 * exp(xb) / 0.0076927) ;  Nota: 1.51714 = exp(120*0.0076927) - 1
    # ln(1-M) = -1.51714 * exp(xb) / 0.0076927

    ln_1_menos_m = -1.51714 * np.exp(xb) / 0.0076927

    return 141.50225 + np.log(-0.00553 * ln_1_menos_m) / 0.090165

In [12]:
df_filt["phenoage"] = calcular_phenoage(df_filt)

In [13]:
df_filt = df_filt.drop(columns=pheno_columns)
print(df_filt.shape)
print()
print(df_filt.columns)

(13251, 53)

Index(['SEQN', 'DMDEDUC2', 'RIAGENDR', 'RIDAGEYR', 'INDFMPIR', 'DR1TCAFF',
       'DR1TSUGR', 'DR1TFIBE', 'DR1TCARB', 'DR1TKCAL', 'DR1TMFAT', 'DR1TPFAT',
       'DR1TSFAT', 'DR1TSODI', 'DR1TPOTA', 'DR1TVC', 'DR1TATOC', 'DR1TPROT',
       'DR1TVB12', 'DR1TCALC', 'DR1TVB6', 'DR2TCAFF', 'DR2TSUGR', 'DR2TFIBE',
       'DR2TCARB', 'DR2TKCAL', 'DR2TMFAT', 'DR2TPFAT', 'DR2TSFAT', 'DR2TSODI',
       'DR2TPOTA', 'DR2TVC', 'DR2TATOC', 'DR2TPROT', 'DR2TVB12', 'DR2TCALC',
       'DR2TVB6', 'BMXBMI', 'BPQ020', 'BPQ030', 'HIQ011', 'PAD680', 'ALQ121',
       'SMQ040', 'SMQ020', 'SLD012', 'SLD013', 'OCD150', 'OCQ180', 'LBDTCSI',
       'CICLO', 'PAMODYR', 'phenoage'],
      dtype='object')


In [ ]:
import os
RAW_PATH = "../data/raw"

def exportar_como_tabela_excel(df: pd.DataFrame, nome_ficheiro: str, nome_folha: str = "Dados") -> None:
    """Exporta um DataFrame formatado como uma Tabela Excel interativa."""
    caminho_completo = os.path.join(RAW_PATH, nome_ficheiro)
    
    with pd.ExcelWriter(caminho_completo, engine="xlsxwriter") as writer:
        df.to_excel(
            writer, sheet_name=nome_folha, startrow=1, header=False, index=False, na_rep="NA"
        )
        workbook = writer.book
        worksheet = writer.sheets[nome_folha]
        (max_row, max_col) = df.shape

        column_settings = [{"header": column} for column in df.columns]

        worksheet.add_table(
            0,
            0,
            max_row,
            max_col - 1,
            {
                "columns": column_settings,
                "style": "Table Style Medium 9",
                "name": "TabelaNHANES_RAW",
            },
        )

        for i, col in enumerate(df.columns):
            largura_maxima = max(df[col].astype(str).map(len).max(), len(str(col))) + 3
            largura_maxima = min(largura_maxima, 50)
            worksheet.set_column(i, i, largura_maxima)

    print(f"Ficheiro '{caminho_completo}' gerado com sucesso!")

In [ ]:
#exportar_como_tabela_excel(df_filt, "nhanes_2017_2023_pheno.xlsx", nome_folha="NHANES")

In [ ]:
#from ydata_profiling import ProfileReport
#prof = ProfileReport(df_filt) 
#prof.to_file(output_file="Dataset_profiling_2.html")

Export report to file: 100%|██████████| 1/1 [00:01<00:00,  1.06s/it]


___
## Dealing with missing values

-> Dietary variables (day 1 and day 2 measurements)

There are three possible scenarios for a given `Dietary` variable measurement:

1. *`Participant X` provided data for both days* (complete data);
2. *`Participant X` provided data for Day 1, but not for Day 2* (partially missing data);
3. *`Participant X` did not provide data for either day* (completely missing data).

> **Note:** If a participant does not provide data for Day 1, there is also no measurement for Day 2, indicating a non-random (monotone) missingness pattern.

In [14]:
df_filt_og = df_filt.copy()

In [15]:
NUTRIENTS = ["CAFF", "SUGR", "FIBE", "CARB", "KCAL", "MFAT", "PFAT", "SFAT", "SODI",
              "POTA", "VC", "ATOC", "PROT", "VB12", "CALC", "VB6"]

for n in NUTRIENTS:
    c1, c2 = f"DR1T{n}", f"DR2T{n}"
    df_filt[f"{n}"] = df_filt[[c1, c2]].mean(axis=1)

#drop the columns for each day
cols_to_drop = [f"DR{d}T{n}" for n in NUTRIENTS for d in (1, 2)]
df_filt = df_filt.drop(columns=cols_to_drop)

In [16]:
cols = [f"{n}" for n in NUTRIENTS]
df_filt[cols] = df_filt[cols].fillna(df_filt[cols].median())

In [17]:
print(df_filt[cols].isna().sum())

CAFF    0
SUGR    0
FIBE    0
CARB    0
KCAL    0
MFAT    0
PFAT    0
SFAT    0
SODI    0
POTA    0
VC      0
ATOC    0
PROT    0
VB12    0
CALC    0
VB6     0
dtype: int64


___
-> Resolving missing values on `INDFMPIR`:

In [18]:
corr = df_filt.corr(numeric_only=True)['INDFMPIR'].drop('INDFMPIR')

top_corr = corr.reindex(corr.abs().sort_values(ascending=False).index)

print(top_corr.index[0], top_corr.iloc[0])

DMDEDUC2 0.4741788135394905


Drop rows were `DMDEDUC2` is 7 (Refused) or 9 (Don't know):

In [19]:
df_filt = df_filt[~df_filt['DMDEDUC2'].isin([7, 9])].copy()

Let's create a binary indicator (INDFMPIR_missing) to flag whether a participant reported their income (1) or not (0).

In [20]:
df_filt['INDFMPIR_reported'] = df_filt['INDFMPIR'].notna().astype(int)

In [21]:
df_filt[['INDFMPIR','INDFMPIR_reported']].head(5)

,INDFMPIR,INDFMPIR_reported
3,5.00,1
8,NaN,0
10,0.83,1
11,1.20,1
19,3.61,1


Now we can use the median per education level (`DMDEDUC2`) to impute the missing values on `INDFMPIR`:

In [22]:
medians_by_educ = df_filt.groupby('DMDEDUC2')['INDFMPIR'].median()

df_filt['INDFMPIR'] = df_filt['INDFMPIR'].fillna(df_filt['DMDEDUC2'].map(medians_by_educ))

In [23]:
print(df_filt['INDFMPIR'].isna().sum())

0


In [24]:
medians_by_educ

DMDEDUC2
1.0    1.19
2.0    1.30
3.0    1.77
4.0    2.43
5.0    4.69
Name: INDFMPIR, dtype: float64

___

-> Resolving missing values on `BMXBMI`:

In [25]:
print(df_filt['BMXBMI'].isna().sum())

194


In [27]:
print(df_filt.shape[0])

13238


Since there is so little rows with missing values for this feature we will be eliminating those:

In [28]:
df_filt = df_filt.dropna(subset=['BMXBMI'])

In [29]:
print(df_filt.shape[0])

13044


___

> Dealing with `Questionnaire` variables:

Making codes 7, 9, 77, 99, 7777, 9999, 77777, 99999 -> Nan

-> `HIQ011` - codes: 7, 9 

In [30]:
df_filt = df_filt[~df_filt['HIQ011'].isin([7, 9])].copy() 

In [31]:
print(df_filt.shape[0]) # dropped only 20 rows, not bad!

13024


-> `PDA680` - codes: 7777, 9999 

In [ ]:
df_filt = df_filt[~df_filt['PAD680'].isin([7777, 9999])].dropna(subset=['PAD680']).copy()
# small sample of missing values, don't impute

In [33]:
df_filt.loc[df_filt['PAD680'] < 60, 'PAD680'] = 60 
# because no one sits for as little as 4 minutes for the entire day

In [34]:
# If not including the sleeping hours (~ 6h) values greater than 1080 (60*18) are also not plausible
# Therefore...

df_filt.loc[df_filt['PAD680'] > 1080, 'PAD680'] = 1080 

In [ ]:
print(df_filt.shape[0]) # dropped only 92 rows, not bad!

12932


-> `PAMODYR` - codes: 7777, 9999

In [39]:
df_filt.loc[df_filt['PAMODYR'].isin([7777,9999]), 'PAMODYR'] = np.nan 

In [41]:
print(df_filt['PAMODYR'].isna().sum())

5520


In [42]:
# We assume that no one trains more than 3 times a day consistently, meaning we are not 
# considering values higher than 1095 (3*365)
df_filt.loc[df_filt['PAMODYR'] > 1095, 'PAMODYR'] = 1095

In [45]:
print(df_filt.groupby('CICLO')['PAMODYR'].apply(lambda s: s.isna().sum()))

CICLO
2017-2020    4490
2021-2023    1030
Name: PAMODYR, dtype: int64


In [46]:
# According to NHANES (Cycle 2017-2020) the participantes who responded "No" (1) to a prior 
# question (PAQ665) have a "Nan" value on PAMODYR. Even though we haven't got access to the same variable in 2021-2023 cycle,
# we will assume the same is happening.
# Of course that mistypes may occur, but for simplicity we will impute 0 on "Nan".
df_filt['PAMODYR'] = df_filt['PAMODYR'].fillna(0) 

-> `BPQ030` - codes: 7, 9 